# Exercise 7 - GANs

> **GPU: Runtime -> Change runtime type -> T4 GPU.** About 6 minutes.

Dataset: **FashionMNIST at 32x32** rather than the lesson's MNIST - same shapes, harder textures, and
mode collapse is easier to spot (does it generate all 10 garment types, or only trousers?).

Eight tasks. Task 4 is a broken training loop with three bugs - the GAN equivalent of chapter 4's
task 5, and every bug in it is one people really write.

In [ ]:
import time, math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device', device)
if device.type != 'cuda':
    print('*** no GPU: switch the runtime, or reduce EPOCHS ***')

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DATA_DIR = '/content/data' if IN_COLAB else './data'
NUM_WORKERS = 2 if IN_COLAB else 0

def set_seed(seed=0):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(0)
torch.backends.cudnn.benchmark = True
plt.rcParams['figure.dpi'] = 110

IMG_SIZE, Z_DIM, BATCH, N_CLASSES = 32, 100, 128, 10

def to_img(t):
    return ((t.detach().cpu() + 1) / 2).clamp(0, 1)

def show_grid(t, nrow=8, title='', figsize=(7, 7)):
    g = make_grid(to_img(t), nrow=nrow, padding=2)
    plt.figure(figsize=figsize); plt.imshow(g.permute(1, 2, 0).numpy(), cmap='gray')
    plt.axis('off'); plt.title(title); plt.show()

print('setup ok - now build the transform in task 1')

---
## Task 1 - Data in the right range

Build `tf`: resize to 32, to tensor, and normalize so the pixel range is **[-1, 1]**.

Then answer in the markdown cell: what breaks if you leave the data in [0, 1] but keep a `tanh`
generator?

In [ ]:
# TODO: tf = transforms.Compose([...])

train_ds = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=tf)
loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                    pin_memory=device.type == 'cuda', drop_last=True)
CLASSES = train_ds.classes

xb, yb = next(iter(loader))
assert xb.shape == (BATCH, 1, 32, 32), f'batch shape {tuple(xb.shape)}'
assert xb.min() < -0.9, f'min is {xb.min():.3f} - data must reach -1, not 0'
assert xb.max() > 0.9, f'max is {xb.max():.3f}'
assert abs(float(xb.mean())) < 0.5, 'data should be roughly centred on 0'
print(f'PASS  range ({xb.min():.2f}, {xb.max():.2f}) | mean {xb.mean():+.3f} | {len(loader)} batches')
show_grid(xb[:32], nrow=8, title='real FashionMNIST', figsize=(6, 3.4))

**What breaks with data in [0,1] and a tanh generator?** ...

---
## Task 2 - The DCGAN generator

`(N, Z_DIM)` noise -> `(N, 1, 32, 32)` image in [-1, 1].

Requirements: four `ConvTranspose2d` layers (`4,1,0` then `4,2,1` three times), `BatchNorm2d` +
`ReLU` after all but the last, `Tanh` at the end, `bias=False` everywhere.

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM, base=64):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, z):
        # TODO: remember z arrives as (N, z_dim) and ConvTranspose2d needs (N, z_dim, 1, 1)
        raise NotImplementedError


G = Generator().to(device)
with torch.no_grad():
    out = G(torch.randn(4, Z_DIM, device=device))

assert out.shape == (4, 1, 32, 32), f'output {tuple(out.shape)} should be (4, 1, 32, 32)'
assert out.min() >= -1.0 and out.max() <= 1.0, 'output must be within [-1, 1] - is Tanh the last layer?'
assert out.abs().max() > 0.3, 'output is suspiciously close to zero everywhere'
n_bn = sum(1 for m in G.modules() if isinstance(m, nn.BatchNorm2d))
assert n_bn == 3, f'{n_bn} BatchNorm2d layers - expected 3 (all but the output layer)'
assert all(m.bias is None for m in G.modules() if isinstance(m, nn.ConvTranspose2d)), 'use bias=False'
print(f'PASS  {sum(p.numel() for p in G.parameters()):,} parameters, output {tuple(out.shape)} '
      f'in ({out.min():.2f}, {out.max():.2f})')

---
## Task 3 - The discriminator

`(N, 1, 32, 32)` -> `(N, 1)` **logits**.

Requirements: `LeakyReLU(0.2)`, **no BatchNorm on the first layer**, no sigmoid, `bias=False`.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, base=64):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO: return (N, 1)
        raise NotImplementedError


D = Discriminator().to(device)
with torch.no_grad():
    logits = D(xb.to(device))

assert logits.shape == (BATCH, 1), f'output {tuple(logits.shape)} should be (N, 1)'
assert logits.abs().max() > 1e-4, 'all-zero output?'
assert not any(isinstance(m, nn.Sigmoid) for m in D.modules()), 'no sigmoid - use BCEWithLogitsLoss'
assert any(isinstance(m, nn.LeakyReLU) for m in D.modules()), 'use LeakyReLU in D'
first_conv_then_bn = list(D.modules())[2:4]
assert not isinstance(first_conv_then_bn[1], nn.BatchNorm2d), 'no BatchNorm directly after the FIRST conv'
print(f'PASS  {sum(p.numel() for p in D.parameters()):,} parameters, logits {tuple(logits.shape)}')

bce = nn.BCEWithLogitsLoss()
set_seed(0)
Gp, Dp = Generator().to(device), Discriminator().to(device)
with torch.no_grad():
    fake = Gp(torch.randn(BATCH, Z_DIM, device=device))
    d0 = (bce(Dp(xb.to(device)), torch.ones(BATCH, 1, device=device))
          + bce(Dp(fake), torch.zeros(BATCH, 1, device=device))).item()
print(f'      initial d_loss {d0:.4f} vs 2ln2 = {2 * math.log(2):.4f} '
      f'({"close" if abs(d0 - 2 * math.log(2)) < 0.6 else "check your init"})')

---
## Task 4 - Find and fix three bugs

`train_gan_buggy` below runs and produces nothing but noise. **Three bugs.** Read it before running.

Hints in the order they appear: what does the D step do to G's weights? Which labels does the
generator's loss use? And what is `opt_G.zero_grad()` clearing at that point in the loop?

In [ ]:
def train_gan_buggy(epochs=1):
    set_seed(0)
    G, D = Generator().to(device), Discriminator().to(device)
    opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        for real, _ in loader:
            real = real.to(device)
            n = real.size(0)
            ones = torch.ones(n, 1, device=device)
            zeros = torch.zeros(n, 1, device=device)
            fake = G(torch.randn(n, Z_DIM, device=device))

            opt_D.zero_grad()
            d_loss = bce(D(real), ones) + bce(D(fake), zeros)
            d_loss.backward()
            opt_D.step()

            g_loss = -bce(D(fake), zeros)
            g_loss.backward()
            opt_G.zero_grad()
            opt_G.step()
    return G, D

**The three bugs are:**

1. ...
2. ...
3. ...

In [ ]:
def train_gan(epochs=6, lr=2e-4, real_label=0.9, log=True):
    """Corrected loop. Return (G, D, history) where history has d_loss, g_loss, acc_real, acc_fake."""
    # TODO
    raise NotImplementedError


EPOCHS = 6
G, D, hist = train_gan(EPOCHS)

assert set(hist) >= {'d_loss', 'g_loss', 'acc_real', 'acc_fake'}, f'history keys: {set(hist)}'
assert len(hist['d_loss']) == EPOCHS
assert isinstance(hist['d_loss'][0], float), 'log floats, not tensors'
assert hist['acc_fake'][-1] > 0.05, 'D is never fooled - G learned nothing, or D is far too strong'
assert hist['acc_real'][-1] > 0.5, 'D cannot even recognise real images'
G.eval()
with torch.no_grad():
    samples = G(torch.randn(64, Z_DIM, device=device))
assert samples.std().item() > 0.2, 'samples have almost no variation - collapsed or untrained'
print(f'PASS  final: d_loss {hist["d_loss"][-1]:.3f} | D correct on real {hist["acc_real"][-1]:.3f} '
      f'| D fooled {hist["acc_fake"][-1]:.3f}')

show_grid(samples, nrow=8, title=f'64 samples after {EPOCHS} epochs')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(hist['d_loss'], marker='o', label='D'); axes[0].plot(hist['g_loss'], marker='s', label='G')
axes[0].axhline(2 * math.log(2), ls=':', c='k', label='2ln2'); axes[0].set_title('losses')
axes[1].plot(hist['acc_real'], marker='o', label='D correct on real')
axes[1].plot(hist['acc_fake'], marker='s', label='D fooled by fake')
axes[1].axhline(0.5, ls=':', c='k'); axes[1].set_title('the useful diagnostic')
for ax in axes:
    ax.set_xlabel('epoch'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()

---
## Task 5 - Measure diversity (detect mode collapse)

Write `diversity(batch)` returning `(mean_pairwise_L2, mean_per_pixel_std)`. Exclude the zero
diagonal from the pairwise mean.

Then compare: real data, your generator, and a deliberately collapsed generator (one sample
repeated). Report the fake/real ratio.

In [ ]:
def diversity(batch):
    """-> (mean pairwise L2 distance between samples, mean per-pixel std across the batch)"""
    # TODO
    raise NotImplementedError


real_batch = next(iter(loader))[0].to(device)
G.eval()
with torch.no_grad():
    fake_batch = G(torch.randn(real_batch.size(0), Z_DIM, device=device))
collapsed = fake_batch[:1].repeat(real_batch.size(0), 1, 1, 1)

dr, sr = diversity(real_batch)
df, sf = diversity(fake_batch)
dc, sc = diversity(collapsed)

assert dc < 1e-4, f'a repeated sample must have ~0 pairwise distance, got {dc:.6f}'
assert dr > 0 and df > 0
print(f'{"":18} {"pairwise":>10} {"px std":>9}')
print(f'{"real":18} {dr:10.3f} {sr:9.4f}')
print(f'{"generated":18} {df:10.3f} {sf:9.4f}')
print(f'{"fully collapsed":18} {dc:10.3f} {sc:9.4f}')
print(f'\ndiversity ratio {df / dr:.3f}')
assert df / dr > 0.4, f'ratio {df / dr:.3f} is low - likely partial mode collapse'
print('PASS')

---
## Task 6 - Which classes did it learn?

Partial mode collapse is the common case: the generator covers some garment types and silently drops
others. Detect it **without labels on the fakes** by training nothing new - use a classifier.

1. Train a small CNN classifier on real FashionMNIST for 2 epochs (>0.85 val accuracy is plenty).
2. Classify 1000 generated samples.
3. Compare the predicted class histogram against the real (uniform) one.

Report which classes are over- and under-represented.

In [ ]:
def train_classifier(epochs=2):
    """-> a trained CNN mapping (N,1,32,32) in [-1,1] to 10 logits, plus its train accuracy."""
    # TODO
    raise NotImplementedError


clf, clf_acc = train_classifier()
print(f'classifier train accuracy {clf_acc:.4f}')
assert clf_acc > 0.85, f'classifier only reached {clf_acc:.3f} - it needs to be trustworthy'

# TODO: classify 1000 generated samples, build the histogram, compare with uniform 0.1
#       store the (10,) fraction array in `gen_hist`

assert gen_hist.shape == (10,) and abs(gen_hist.sum() - 1.0) < 1e-6
print(f'\n{"class":16} {"generated":>10} {"real":>7}')
for i, c in enumerate(CLASSES):
    flag = '  <- under' if gen_hist[i] < 0.05 else ('  <- over' if gen_hist[i] > 0.18 else '')
    print(f'{c:16} {gen_hist[i]:10.3f} {0.100:7.3f}{flag}')
print(f'\nmax deviation from uniform: {np.abs(gen_hist - 0.1).max():.3f}')
print(f'classes essentially missing (<2%): {[CLASSES[i] for i in range(10) if gen_hist[i] < 0.02]}')

**Is your generator covering all ten classes? Which are weakest, and why might those be?** ...

---
## Task 7 - Latent interpolation, both ways

Implement `interpolate(G, z_a, z_b, steps, spherical)` supporting **both** linear and spherical
interpolation, and plot the two side by side for the same endpoints.

Then explain why the linear midpoint looks washed out.

In [ ]:
@torch.no_grad()
def interpolate(G, z_a, z_b, steps=10, spherical=True):
    """z_a, z_b: (Z_DIM,) tensors -> (steps, 1, 32, 32) generated images."""
    # TODO
    raise NotImplementedError


set_seed(5)
za, zb = torch.randn(Z_DIM, device=device), torch.randn(Z_DIM, device=device)
lin = interpolate(G, za, zb, 10, spherical=False)
sph = interpolate(G, za, zb, 10, spherical=True)

assert lin.shape == (10, 1, 32, 32) and sph.shape == (10, 1, 32, 32)
assert torch.allclose(lin[0], sph[0], atol=1e-4), 'both should start at G(z_a)'
assert not torch.allclose(lin[5], sph[5], atol=1e-3), 'the midpoints should differ'
print(f'PASS')
print(f'  |z| endpoints: {za.norm():.2f}, {zb.norm():.2f}  (expected around sqrt({Z_DIM}) = {math.sqrt(Z_DIM):.1f})')
lin_mid = ((1 - 0.5) * za + 0.5 * zb).norm()
print(f'  |linear midpoint| {lin_mid:.2f}  <- notice it is SMALLER than the endpoints')

fig, axes = plt.subplots(2, 10, figsize=(13, 2.9))
for c in range(10):
    axes[0, c].imshow(to_img(lin[c])[0].numpy(), cmap='gray')
    axes[1, c].imshow(to_img(sph[c])[0].numpy(), cmap='gray')
for ax in axes.ravel(): ax.axis('off')
axes[0, 0].set_ylabel('linear'); axes[1, 0].set_ylabel('spherical')
plt.suptitle('linear (top) vs spherical (bottom) interpolation')
plt.tight_layout()

**Why is the linear midpoint off-distribution?** ...

---
## Task 8 - Conditional GAN

Build `CondGenerator` and `CondDiscriminator` that both take the label, train for 6 epochs, and
produce a 10x10 grid where **row `i` is garment class `i`**.

`D` must receive the label too. Say in the markdown cell what happens if it doesn't.

In [ ]:
class CondGenerator(nn.Module):
    def __init__(self, z_dim=Z_DIM, base=64, emb=32):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, z, y):
        # TODO
        raise NotImplementedError


class CondDiscriminator(nn.Module):
    def __init__(self, base=64):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x, y):
        # TODO
        raise NotImplementedError


def train_cgan(epochs=6, lr=2e-4, real_label=0.9):
    """-> (cG, history)"""
    # TODO
    raise NotImplementedError


cG, chist = train_cgan(6)

with torch.no_grad():
    probe_y = torch.arange(N_CLASSES, device=device)
    probe_z = torch.randn(1, Z_DIM, device=device).repeat(N_CLASSES, 1)
    probe = cG(probe_z, probe_y)
assert probe.shape == (N_CLASSES, 1, 32, 32)
pred = clf(probe).argmax(1).cpu().numpy()
match = (pred == np.arange(N_CLASSES)).mean()
print(f'classifier agrees with the requested class {match * 100:.0f}% of the time (10 samples)')
assert match >= 0.4, f'only {match:.1f} - is D receiving the label?'
print('PASS')

cG.eval()
with torch.no_grad():
    ys = torch.arange(N_CLASSES, device=device).repeat_interleave(10)
    zs = torch.randn(10, Z_DIM, device=device).repeat(N_CLASSES, 1)
    grid = cG(zs, ys)
g = make_grid(to_img(grid), nrow=10, padding=2)
plt.figure(figsize=(8, 8)); plt.imshow(g.permute(1, 2, 0).numpy(), cmap='gray'); plt.axis('off')
plt.title('row i = requested class i (columns share a latent z)')
plt.show()
for i, c in enumerate(CLASSES):
    print(f'  row {i}: {c}')

**What happens if only G sees the label and D doesn't?** ...

---
## Done

- [ ] I can write both GAN losses with the right labels, from memory.
- [ ] I know why the fake batch is detached in the D step.
- [ ] I know that GAN loss curves are not a progress signal, and what to watch instead.
- [ ] I can detect mode collapse quantitatively, including the partial kind.

Solutions: [`solutions/sol07_gan.ipynb`](solutions/sol07_gan.ipynb)